In [ ]:
import textgrid
import math
import cv2
import numpy as np
import os
from tqdm import tqdm

In [ ]:
user_name = 'astitva'

# set paths
ALIGNER_ROOT = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/ForcedAligner'
ASSETS_ROOT = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/PresetGeneration/OUTPUT/DRAWINGS'
VIDEO_SAVE_DIR = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_RESULTS'
FFMPEG_PATH = f'/mnt/users_scratch/{user_name}/WORKSPACE/ffmpeg/installation/bin'

In [ ]:
# helper function for compositing assets
def composite(base, mouth, eyes, use_default_mouth=False, use_default_eyes=False):
    mask_mouth = mouth[:,:,3]/255
    mask_eyes = eyes[:,:,3]/255
    mask = mask_mouth + mask_eyes
    if use_default_eyes:
      mask = mask_mouth
    if use_default_mouth:
        mask = mask_eyes
    mask_im = np.repeat(mask[..., np.newaxis], 3, axis=2)
    mask_mouth_im = np.repeat(mask_mouth[..., np.newaxis], 3, axis=2)
    mask_eyes_im = np.repeat(mask_eyes[..., np.newaxis], 3, axis=2)
    composited = base*(1-mask_im) 
    if not use_default_eyes:
        composited += mask_eyes_im*eyes[:,:,:3]
    if not use_default_mouth:
        composited += mask_mouth_im*mouth[:,:,:3]
    return composited.astype('uint8')

# ANIMATION : MOUTH + EYES 

### DEFINE PHONE-VISEME MAPPING

In [ ]:
phoneme2viseme = {
    'AA':2,
    'AE':7, 
    'AH':16,
    'AO':8,
    'AW':8,
    'AY':4,
    'AX':2,
    'B':1, 
    'CH':14,
    'D':1,
    'DH':14,
    'EH':7,
    'ER':7,
    'EY':2,
    'F':15,
    'G':11,
    'HH':11,
    'IH':1,
    'IY':4,
    'JH':14,
    'K':11,
    'L':13,
    'M':1,
    'N':13,
    'NG':11,
    'OW':8,
    'OY':8,
    'P':1,
    'R':13,
    'S':14,
    'SH':10,
    'T':1,
    'TH':14,
    'UH':9,
    'UW':9,
    'UX':9,
    'V':15,
    'W':9,
    'Y':4,
    'Z':12,
    'ZH':12
}

### TEXT-GRID FROM FORCE ALIGNER

In [ ]:
filename = 'babyshark'

tg_path = f'{ALIGNER_ROOT}/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)

words = tg[0]
phonemes = tg[1]

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)

words

### DEFINE EYE-WORD MAPPING

In [ ]:
# eye_word_mapping = {'please':2, 'thankyou':3}
eye_word_mapping = {'mommy':3,'daddy':0,'grandma':6,'grandpa':7, 'hunt':5}


### CREATE ANIMATION

In [ ]:
characters = ['0a3b9f4c787743458c7ca1cc77b902ea']

crop_face = True
fps=240

ext = 'mp4'

suffix = ''
if crop_face:
    suffix = '_face'

skip_start_frames = 0
if filename=='babyshark':
    skip_start_frames = 1063

for character_id in characters:

    MOUTH_ROOT = f'{ASSETS_ROOT}/mouth/{character_id}'
    EYES_ROOT = f'{ASSETS_ROOT}/eyes/{character_id}'
    SAVE_ROOT = f'{VIDEO_SAVE_DIR}/{character_id}/'
    os.makedirs(SAVE_ROOT, exist_ok=True)

    asset_dict = {}
    asset_dict['inpainted_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_mouth_only.png')
    asset_dict['inpainted_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_only.png')
    asset_dict['inpainted_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_mouth.png')
    asset_dict['inpainted_face_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_face_mouth_only.png')
    asset_dict['inpainted_face_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_only.png')
    asset_dict['inpainted_face_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_mouth.png')

    mouth_files = os.listdir(f'{MOUTH_ROOT}/assets/')
    for m in mouth_files:
        asset_dict[m[:-4]] = cv2.imread(f'{MOUTH_ROOT}/assets/{m}', -1)

    eye_files = os.listdir(f'{EYES_ROOT}/assets/')
    for e in eye_files:
        asset_dict[e[:-4]] = cv2.imread(f'{EYES_ROOT}/assets/{e}', -1)

    asset_dict.keys()
    
    base = asset_dict[f'inpainted{suffix}_mouth_only']
    eyes_id = 0
    blink_id = 2
    blink_gap = 2 #seconds
    num_blink_frames = 12
    
    USE_DEFAULT_MOUTH = False
    USE_DEFAULT_EYES = True
    
    video=cv2.VideoWriter(f'{SAVE_ROOT}/{filename}_no_audio.{ext}',cv2.VideoWriter_fourcc(*'mp4v'),fps,(1024,1024))
    buffer = np.ones((1024,1024,3)).astype('uint8')*255
    current_frame_count = 0
    
    for i in tqdm(range(len(words))):
        total_duration = words[i].duration()
        word_frame_count = int(fps*total_duration)
        cumulative_frame_count = 0
    
        #eyes asset
        try:
            eyes_id = eye_word_mapping[words[i].mark]
            # base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
            base = asset_dict[f'inpainted{suffix}_eyes_mouth']
            USE_DEFAULT_EYES=False
        except:
            pass
    
        # eyes = cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{eyes_id}_asset.png', -1)
        eyes = asset_dict[f'eyes_{eyes_id}{suffix}']

        
        for p in words_phonemes[i]:
            frame_count = math.ceil(fps*p.duration())
    
            #default mouth asset
            mouth_id = 1
            # buffer_mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
            buffer_mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
            
            # default
            buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
            
            viseme_id=''
            if p.mark!='':
                try:
                    mouth_id = phoneme2viseme[p.mark[:2]]
                    # mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
                    mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
                    buffer_mouth = mouth
                    buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
                except:
                    pass
            # print(USE_DEFAULT_EYES)
            for _ in range(frame_count):
                if cumulative_frame_count>=word_frame_count:
                    break
                #blink
                if current_frame_count%(fps*blink_gap)<num_blink_frames:
                    # blink_eyes =  cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{blink_id}_asset.png', -1)
                    blink_eyes = asset_dict[f'eyes_{blink_id}{suffix}']
                    # blink_base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
                    blink_base = asset_dict[f'inpainted{suffix}_eyes_mouth']

                    if current_frame_count>skip_start_frames:
                        video.write(composite(blink_base, buffer_mouth, blink_eyes))
                else:
                    if current_frame_count>skip_start_frames:
                        video.write(buffer)
                cumulative_frame_count += 1
                current_frame_count += 1
                
    video.release()

    command = f'{FFMPEG_PATH}/ffmpeg -i {SAVE_ROOT}/{filename}_no_audio.{ext} -i {ALIGNER_ROOT}/inputs/{filename}/{filename}_trimmed.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 {SAVE_ROOT}/{filename}.{ext}'
    os.system(f'rm {SAVE_ROOT}/{filename}.{ext}')
    os.system(command)
    
    print()
    print()
    print(f'Video saved --> {SAVE_ROOT}/{filename}.{ext}')